In [ ]:
"""Analyze rating differences between 1. Liga and Promotion League.

This script compares player ratings across leagues and position groups. It is
used to estimate league-specific performance differences and create adjustment
summaries for the scouting model.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from ml.toolkit.ml_utilities import (
    POSITION_TO_GROUP,
    normalize_league,
    parse_season_start_year,
)


DATA_DIR = PROJECT_ROOT / "data" / "transform"
MIN_MATCHES_PER_SEASON = 5

PAIRS_OUTPUT_PATH = CURRENT_DIR / "league_difference_pairs.csv"
SUMMARY_OUTPUT_PATH = CURRENT_DIR / "league_difference_summary.csv"
STABLE_SUMMARY_OUTPUT_PATH = (
    CURRENT_DIR / "league_difference_stable_position_summary.csv"
)


def load_transform_tables(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the transformed tables needed for the league difference analysis.

    The function checks whether all required CSV files are available and then
    loads player statistics, player information, and match information.
    """
    required_files = {
        "player_stats": data_dir / "player_stats.csv",
        "players": data_dir / "players.csv",
        "matches": data_dir / "matches.csv",
    }

    missing_files = [str(path) for path in required_files.values() if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing transformed files: {missing_files}")

    player_stats = pd.read_csv(required_files["player_stats"])
    players = pd.read_csv(required_files["players"])
    matches = pd.read_csv(required_files["matches"])

    return player_stats, players, matches


def build_match_rating_dataset(
    player_stats: pd.DataFrame,
    players: pd.DataFrame,
    matches: pd.DataFrame,
) -> pd.DataFrame:
    """Create a match-level rating dataset with league and position context.

    The function combines player statistics with match and player information,
    normalizes league names, assigns position groups, converts relevant columns
    to numeric values, and keeps only valid rating records from 1. Liga and
    Promotion League.
    """
    dataset = (
        player_stats.merge(
            matches[["match_id", "season", "league", "date"]],
            on="match_id",
            how="left",
        )
        .merge(
            players[["player_id", "player_name", "position"]],
            on="player_id",
            how="left",
        )
    )

    dataset["league_group"] = dataset["league"].map(normalize_league)
    dataset["season_start_year"] = dataset["season"].map(parse_season_start_year)
    dataset["position_group"] = dataset["position"].map(POSITION_TO_GROUP)
    dataset["rating"] = pd.to_numeric(dataset["rating"], errors="coerce")
    dataset["minutes"] = pd.to_numeric(dataset["minutes"], errors="coerce")

    return dataset[
        dataset["league_group"].isin(["1. Liga", "Promotion League"])
        & dataset["rating"].notna()
        & dataset["season_start_year"].notna()
        & dataset["position_group"].notna()
        & (dataset["minutes"] > 0)
    ].copy()


def aggregate_player_season_ratings(dataset: pd.DataFrame) -> pd.DataFrame:
    """Aggregate match-level ratings into player-season ratings.

    The function groups all valid match ratings by player, season, league, and
    position group. It calculates average ratings, number of matches played, and
    total minutes, then keeps only player-seasons with enough matches.
    """
    player_season_ratings = (
        dataset.groupby(
            [
                "player_id",
                "player_name",
                "season",
                "season_start_year",
                "league_group",
                "position",
                "position_group",
            ],
            as_index=False,
        )
        .agg(
            avg_rating=("rating", "mean"),
            matches_played=("match_id", "nunique"),
            minutes_total=("minutes", "sum"),
        )
    )

    return player_season_ratings[
        player_season_ratings["matches_played"] >= MIN_MATCHES_PER_SEASON
    ].copy()


def build_league_transition_pairs(player_season_ratings: pd.DataFrame) -> pd.DataFrame:
    """Build player-season pairs between 1. Liga and Promotion League.

    The function matches players who appeared in both leagues in neighbouring
    seasons. It calculates the rating difference between the two leagues and
    stores additional information about transition direction and position-group
    stability.
    """
    one_liga = player_season_ratings[
        player_season_ratings["league_group"] == "1. Liga"
    ].copy()

    promotion_league = player_season_ratings[
        player_season_ratings["league_group"] == "Promotion League"
    ].copy()

    one_liga = one_liga.rename(
        columns={
            "season": "season_first_league",
            "season_start_year": "season_year_first_league",
            "avg_rating": "avg_rating_first_league",
            "matches_played": "matches_first_league",
            "minutes_total": "minutes_first_league",
            "position": "position_first_league",
            "position_group": "position_group_first_league",
        }
    )

    promotion_league = promotion_league.rename(
        columns={
            "season": "season_promotion_league",
            "season_start_year": "season_year_promotion_league",
            "avg_rating": "avg_rating_promotion_league",
            "matches_played": "matches_promotion_league",
            "minutes_total": "minutes_promotion_league",
            "position": "position_promotion_league",
            "position_group": "position_group_promotion_league",
        }
    )

    pairs = one_liga.merge(
        promotion_league,
        on=["player_id", "player_name"],
        how="inner",
    )

    pairs = pairs[
        (
            pairs["season_year_promotion_league"]
            - pairs["season_year_first_league"]
        ).abs()
        == 1
    ].copy()

    pairs["rating_diff_promotion_minus_first_league"] = (
        pairs["avg_rating_promotion_league"]
        - pairs["avg_rating_first_league"]
    )

    pairs["transition_direction"] = np.where(
        pairs["season_year_promotion_league"] > pairs["season_year_first_league"],
        "1. Liga -> Promotion League",
        "Promotion League -> 1. Liga",
    )

    pairs["same_position_group"] = (
        pairs["position_group_first_league"]
        == pairs["position_group_promotion_league"]
    )

    return pairs


def summarize_pairs(pairs: pd.DataFrame) -> pd.DataFrame:
    """Summarize league transition pairs by position group.

    The function calculates average, median, and standard deviation of rating
    differences between 1. Liga and Promotion League. It also includes match and
    minutes statistics to support interpretation of the league adjustment.
    """
    return (
        pairs.groupby("position_group_first_league", as_index=False)
        .agg(
            mean_diff=("rating_diff_promotion_minus_first_league", "mean"),
            median_diff=("rating_diff_promotion_minus_first_league", "median"),
            std_diff=("rating_diff_promotion_minus_first_league", "std"),
            pair_count=("rating_diff_promotion_minus_first_league", "count"),
            mean_rating_first_league=("avg_rating_first_league", "mean"),
            mean_rating_promotion_league=("avg_rating_promotion_league", "mean"),
            mean_matches_first_league=("matches_first_league", "mean"),
            mean_matches_promotion_league=("matches_promotion_league", "mean"),
            mean_minutes_first_league=("minutes_first_league", "mean"),
            mean_minutes_promotion_league=("minutes_promotion_league", "mean"),
        )
        .sort_values("mean_diff", ascending=False)
    )


def save_outputs(
    pairs: pd.DataFrame,
    summary: pd.DataFrame,
    stable_summary: pd.DataFrame,
) -> None:
    """Save the league difference analysis outputs as CSV files.

    The function writes the transition pairs, the full summary, and the stable
    position-group summary to disk. It also prints the output paths and a compact
    overview of the most relevant summary values.
    """
    pairs.to_csv(PAIRS_OUTPUT_PATH, index=False)
    summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)
    stable_summary.to_csv(STABLE_SUMMARY_OUTPUT_PATH, index=False)

    print(f"Saved pairs: {PAIRS_OUTPUT_PATH}")
    print(f"Saved summary: {SUMMARY_OUTPUT_PATH}")
    print(f"Saved stable-position summary: {STABLE_SUMMARY_OUTPUT_PATH}")

    print(
        summary[
            ["position_group_first_league", "median_diff", "pair_count"]
        ].to_string(index=False)
    )


def main() -> None:
    """Run the complete league difference analysis.

    The function loads the transformed data, builds the match-level dataset,
    aggregates player-season ratings, creates league transition pairs, summarizes
    the differences, and saves all output files.
    """
    player_stats, players, matches = load_transform_tables(DATA_DIR)

    match_rating_dataset = build_match_rating_dataset(
        player_stats,
        players,
        matches,
    )

    player_season_ratings = aggregate_player_season_ratings(match_rating_dataset)
    pairs = build_league_transition_pairs(player_season_ratings)

    summary = summarize_pairs(pairs)
    stable_summary = summarize_pairs(pairs[pairs["same_position_group"]].copy())

    save_outputs(pairs, summary, stable_summary)


if __name__ == "__main__":
    main()